In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

STATING OUR UNIVERSE

In [2]:
universe = [
    "NVDA",  # NVIDIA
    "AAPL",  # Apple
    "MSFT",  # Microsoft
    "GOOGL", # Alphabet
    "AMZN",  # Amazon
    "META"   # Meta
]

In [3]:
universe = ['AAPL', 'JPM', 'JNJ', 'XOM', 'WMT', 'CAT']

## Why Log Returns Instead of Close Prices or Simple Returns

### Why not raw close prices?

Close prices are **non-stationary** — they trend upward or downward over time and their statistical properties (mean, variance) change depending on the time period you look at. Almost every calculation in this project — correlation, volatility, dispersion — assumes the underlying series is *roughly stationary* (fluctuating around a stable level, not systematically drifting). Feeding raw prices into these calculations produces misleading results: two stocks that are simply both trending upward over years would appear artificially "correlated," even if their day-to-day movements are unrelated.

### Why not simple returns (`(P_t - P_t-1) / P_t-1`)?

Simple returns are a valid alternative and widely used, but log returns have two properties that make them the better default for this kind of analysis:

1. **Additivity across time.** Log returns can be summed across multiple periods to get the total return over that period:
   `log_return(day1) + log_return(day2) = log_return(2-day period)`
   Simple returns don't have this property — you have to multiply (1 + r) terms instead, which is less convenient for rolling-window calculations like the ones used throughout this project (20-day volatility, 60-day correlation windows, etc.).

2. **Symmetry.** A stock that goes up 10% one day and down 10% the next does **not** return to its starting price under simple returns (it ends up slightly lower), which distorts volatility and return calculations. Log returns treat symmetric percentage moves symmetrically, which better matches the statistical assumptions used in the covariance, correlation, and eigenvalue calculations throughout this pipeline.

### Practical reason for this project specifically

Every core calculation — correlation matrices, eigenvalue decomposition, rolling volatility, cross-sectional dispersion — is mathematically defined in terms of **returns**, not prices. Log returns are the standard convention in this type of quantitative/RMT-style analysis (matching the source paper's own methodology), making results directly comparable and consistent with established practice in the field.

COMPUTING LOG RETURNS


In [4]:
def log_returns_data(universe):

  data=pd.DataFrame()

  for i in universe:
    ticker=yf.Ticker(i)
    df=ticker.history(period='10y')
    name=f"log_returns_{i}"
    data[name]=np.log(df['Close']/df['Close'].shift(1))

  return data.dropna()

In [5]:
data=log_returns_data(universe)
data.tail()

,log_returns_AAPL,log_returns_JPM,log_returns_JNJ,log_returns_XOM,log_returns_WMT,log_returns_CAT
Date,,,,,,
2026-08-31 00:00:00-04:00,-0.008955,-0.004484,-0.008204,0.026697,0.017119,-0.003480
2026-09-01 00:00:00-04:00,0.025797,-0.003010,0.019887,0.022121,0.009963,-0.023228
2026-09-02 00:00:00-04:00,-0.000523,0.003572,0.014715,-0.002434,0.001604,0.016699
2026-09-03 00:00:00-04:00,0.009952,0.016261,0.011632,-0.011889,0.021725,0.009872
2026-09-04 00:00:00-04:00,-0.025426,-0.009491,-0.011560,-0.017036,-0.011876,0.017100


In [6]:
stock_cols=data.columns


 ---



## Why Eigenvalues (of the Correlation Matrix)

### The problem eigenvalues solve

A correlation matrix on its own is hard to summarize. For a 6-stock universe, it's a 6×6 table of pairwise correlation numbers — but there's no single number in that table that tells you "how concentrated is the co-movement across the whole market right now." Eigenvalue decomposition solves exactly this: it compresses the entire matrix into a small set of numbers that describe its overall *shape* or *structure*, rather than requiring you to inspect every pairwise entry individually.

### What the leading eigenvalue actually measures

The **leading (largest) eigenvalue** measures how much of the total co-movement in the matrix is explained by a single, dominant pattern:

- **Calm markets:** correlations are mixed — some pairs move together, some don't, no single dominant pattern. The eigenvalues are relatively spread out, and the leading eigenvalue is only moderately large.
- **Crisis/stressed markets:** correlations collapse toward "everything moves together" — nearly every stock reacts to the same shock (panic selling, liquidity stress, macro shock). This shows up mathematically as one eigenvalue becoming very large (approaching the total number of stocks, since correlation-1 across all pairs is the theoretical maximum), while the rest shrink.

So the leading eigenvalue is a direct, quantitative signature of "is the market currently behaving like one thing, or many independent things" — which is precisely the condition that causes diversification to fail.

### Why the second eigenvalue matters too

The paper this project draws on documents that the **second eigenvalue** tends to move in the *opposite* direction — shrinking when the leading eigenvalue spikes, and vice versa. This counter-cyclical relationship (which was independently confirmed on this project's own data, showing a correlation of approximately -0.62 between the two) makes the *ratio* of leading-to-second eigenvalue a sharper, more discriminating regime signal than either number alone.

### Why the eigenvector matters, not just the eigenvalue

The eigenvalue tells you *that* a dangerous co-movement pattern exists; the corresponding **eigenvector** tells you *which specific stocks* are driving that pattern. This is what makes the signal actionable — rather than just knowing "the market looks risky," the eigenvector identifies exactly which holdings in the portfolio are contributing most to that risk, enabling a targeted adjustment (reducing exposure to those specific stocks) rather than a blanket, uninformed reaction.

### In short

Eigenvalues turn an unwieldy correlation matrix into a small number of interpretable signals that directly measure the specific phenomenon this project cares about — the breakdown of diversification during market stress — while the corresponding eigenvector translates that signal into a concrete, stock-level action.

COMPUTING EIGENVALUES

In [7]:
def compute_rolling_eigenvalues(data, window=60):
    leading_eigenvalues = []
    second_eigenvalues = []
    third_eigenvalues = []   # <-- new
    dates = []

    for i in range(window, len(data)):
        window_data = data.iloc[i-window:i]
        corr_matrix = window_data.corr()

        eigenvalues = np.linalg.eigvalsh(corr_matrix)
        eigenvalues_sorted = eigenvalues[::-1]

        leading_eigenvalues.append(eigenvalues_sorted[0])
        second_eigenvalues.append(eigenvalues_sorted[1])
        third_eigenvalues.append(eigenvalues_sorted[2])   # <-- new
        dates.append(data.index[i])

    eigen_df = pd.DataFrame({
        'leading_eigenvalue': leading_eigenvalues,
        'second_eigenvalue': second_eigenvalues,
        'third_eigenvalue': third_eigenvalues   # <-- new
    }, index=dates)

    eigen_df['eigenvalue_ratio'] = eigen_df['leading_eigenvalue'] / eigen_df['second_eigenvalue']
    eigen_df.index = eigen_df.index.tz_localize(None)


    return eigen_df


eigen_df=compute_rolling_eigenvalues(data,60)

## Why `add_vol_desperssion_vix(n)` — Purpose of This Function

### What it does, in one line

Takes the log-returns dataframe and adds the volatility, dispersion, and regime-label columns needed for classification — turning raw per-stock returns into the market-wide signals the rest of the pipeline depends on.

### Why each piece inside it exists

**`market_realized_vol_20d`** — measures how turbulent the overall market has been recently. Computed as the rolling 20-day standard deviation of the average return across all stocks, annualized (`* sqrt(252)`) to match standard volatility conventions. This answers: *"is the market moving a lot right now, or is it calm?"*

**`market_cross_sectional_dispersion`** — measures how differently stocks are behaving from each other *on a single day*, as opposed to how much the market as a whole is moving. Computed as the standard deviation *across stocks* (row-wise), not across time. This captures a genuinely different signal than volatility: a market can crash with every stock falling together (low dispersion, high volatility) or stay flat overall while individual stocks diverge sharply (high dispersion, low volatility) — these are different risk conditions worth distinguishing.

**The regime labels (`calm` / `transitioning` / `stressed`)** — generated by thresholding a volatility-derived signal (originally market_realized_vol_20d directly, later replaced by a model-predicted VIX proxy for cleaner, less noisy boundaries) into three percentile-based buckets. These labels become the target variable the classifier is trained to predict.

### Why this function exists as a single unit, rather than separate steps

Volatility, dispersion, and the regime label are all downstream of the same underlying calculation — the market-wide average return — so computing them together in one function keeps the pipeline consistent and avoids recomputing shared intermediate values (like the average return series) multiple times across the notebook.

### Why "vix" is in the function name despite not using real VIX

An earlier version of this function pulled the real market VIX index to generate labels. This was replaced (for reasons of avoiding label leakage and universe/index mismatch — see the note on VIX-based labeling elsewhere in this project) with a self-contained, universe-derived volatility signal instead. The function name is a legacy artifact from that earlier version and no longer reflects an external VIX dependency.

In [8]:
def add_vol_desperssion_vix(n):
  n['market_realized_vol_20d']=n.mean(axis=1).rolling(20).std()*np.sqrt(252)
  n['market_cross_sectional_dispersion']=n.std(axis=1)
  n['VIX']=n.mean(axis=1).rolling(20).std()

  return n

ndata=add_vol_desperssion_vix(data.copy())

In [9]:
def final(e,n):
  final_data=n.merge(e,how='inner',right_index=True,left_index=True)
  return final_data

# Ensure ndata's index is timezone-naive before merging
ndata.index = ndata.index.tz_localize(None)
final_data=final(eigen_df.copy(),ndata.copy())

## Why `vix_model` — Purpose of This Model

### What it does, in one line

Predicts VIX (a proxy for overall market stress) from this project's own engineered features — used exclusively to generate cleaner, more reliable regime labels for training the actual classifier, not used as a live feature or a final output.

### Why this model exists at all

Two earlier labeling approaches were tried first, and both had real problems:

1. **Thresholding real VIX directly** — VIX is derived from S&P 500 options, a broad, ~500-stock index. Using it to label a small, single-sector stock universe created a genuine mismatch: VIX reflects broad-market stress that doesn't necessarily correspond to stress in this specific universe, and vice versa.

2. **Thresholding `market_realized_vol_20d` directly** — a single raw feature, computed only from this project's own stocks. This avoided the universe-mismatch problem, but produced noisier, less discriminating regime boundaries (~57–68% downstream classifier accuracy), since one raw signal alone doesn't capture the full picture of market stress.

`vix_model` was built to get the best of both: a model trained to predict VIX-*like* stress levels using this project's own multi-feature inputs (volatility, dispersion, eigenvalue structure). The result (`pred_vix`) is a smoothed, multi-feature-consistent stress signal — used only to define regime label thresholds, producing meaningfully cleaner boundaries (accuracy improved to ~85–90% once this approach was adopted).

### Why `vix_model` is trained only on the training-period rows

To avoid leaking information from the test period into the labels used for training the main classifier, `vix_model` is fit strictly on the training split (`data[:1800]`), then applied to generate predictions across the full dataset — including test rows it never saw during fitting. This preserves the trustworthiness of later test-set accuracy checks.

### Why `pred_vix` is never used as a feature in the main classifier

Since `pred_vix` is deliberately trained to approximate the very signal used to generate the labels, including it as an input feature to the classifier would let the classifier partially "see" its own answer key — a subtler form of the same target-leakage problem that occurred when real VIX was used as a feature. It is used exclusively to build the `regime` label column, and dropped from the feature set used for actual classification.

In [10]:
final_data['vix_-1']=final_data['VIX'].shift(-1)
final_data.dropna(inplace=True)

In [11]:
def vix_model(data):
  data.drop((data.select_dtypes(include='object').columns),axis=1,inplace=True)

  from sklearn.model_selection import train_test_split as tts , GridSearchCV as grid ,RandomizedSearchCV as rand
  from sklearn.linear_model import LinearRegression
  from sklearn.ensemble import RandomForestRegressor,VotingRegressor
  from xgboost import XGBRegressor
  from sklearn.neighbors import KNeighborsRegressor
  from sklearn.naive_bayes import GaussianNB
  from sklearn.metrics import accuracy_score,classification_report,r2_score
  from sklearn.tree import DecisionTreeRegressor
  from lightgbm import LGBMRegressor
  train=data[:1800]
  test=data[1800:]

  x=train.drop(['VIX','vix_-1'],axis=1)
  y=train['VIX']

  xtrain,xtest,ytrain,ytest=tts(x,y,shuffle=False,test_size=0.2)

  model=VotingRegressor([
    ('rfr',RandomForestRegressor(random_state=42)),
    ('xgb',XGBRegressor(random_state=42)),
    ('lr',LinearRegression()),
  ])
  model.fit(xtrain,ytrain)

  pred=model.predict(data.drop(['VIX','vix_-1'],axis=1))

  return pred

col=vix_model(final_data)
col

array([0.00680434, 0.00672442, 0.00644262, ..., 0.00645331, 0.00607439,
       0.00629192])

In [12]:
final_data['pred_vix']=col
final_data.drop(['VIX','vix_-1'],axis=1,inplace=True)

In [13]:
lower=final_data['pred_vix'].quantile(0.33)
upper=final_data['pred_vix'].quantile(0.66)

l=[]
for i in final_data['pred_vix']:
  if i<=lower:
    l.append('calm')
  elif lower<i<=upper:
    l.append('transitioning')
  else:
    l.append('stressed')
final_data['regime']=l

import numpy as np
import pandas as pd

# --- Third eigenvalue (requires modifying compute_rolling_eigenvalues to also store it) ---
# If not already done, add this inside that function's loop:
#     third_eigenvalues.append(eigenvalues_sorted[2])
# and this to eigen_df construction:
#     eigen_df['third_eigenvalue'] = third_eigenvalues
# Then it flows into final_data automatically via the merge you already did.


# --- Rolling slope helper ---
def rolling_slope(series, window=10):
    return series.rolling(window).apply(
        lambda x: np.polyfit(range(len(x)), x, 1)[0], raw=True
    )
# Rate-of-change features — capture the dynamics of regime shift, not just the level
final_data['eigenvalue_ratio_diff5'] = final_data['eigenvalue_ratio'].diff(5)
final_data['vol_diff5'] = final_data['market_realized_vol_20d'].diff(5)
final_data['dispersion_diff5'] = final_data['market_cross_sectional_dispersion'].diff(5)

# diff() introduces NaN for the first 5 rows (no prior value to compare agains
# --- New features ---
final_data['eigenvalue_ratio_slope10'] = rolling_slope(final_data['eigenvalue_ratio'], 10)
final_data['vol_slope10'] = rolling_slope(final_data['market_realized_vol_20d'], 10)
final_data['eigenvalue_gap_2_3'] = final_data['second_eigenvalue'] - final_data['third_eigenvalue']
final_data['vol_of_vol_10d'] = final_data['market_realized_vol_20d'].rolling(10).std()
final_data['market_return'] = final_data.select_dtypes(include=np.number).mean(axis=1)
final_data['market_return_skew_20d'] = final_data['market_return'].rolling(20).skew()
# 3. Rolling correlation between volatility and dispersion — divergence can signal transition
final_data['vol_dispersion_corr_20d'] = final_data['market_realized_vol_20d'].rolling(20).corr(
    final_data['market_cross_sectional_dispersion']
)

# 4. Return autocorrelation — crashes/panics often show negative autocorrelation (sharp reversals)
final_data['return_autocorr_20d'] = final_data['market_return'].rolling(20).apply(
    lambda x: pd.Series(x).autocorr(lag=1), raw=False
)

# 5. Rolling z-score of today's volatility vs its own recent history — "how unusual is today specifically"
final_data['vol_zscore_60d'] = (
    final_data['market_realized_vol_20d'] - final_data['market_realized_vol_20d'].rolling(60).mean()
) / final_data['market_realized_vol_20d'].rolling(60).std()

# 6. Dispersion z-score, same idea, for cross-sectional dispersion
final_data['dispersion_zscore_60d'] = (
    final_data['market_cross_sectional_dispersion'] - final_data['market_cross_sectional_dispersion'].rolling(60).mean()
) / final_data['market_cross_sectional_dispersion'].rolling(60).std()

cum_return = final_data['market_return'].cumsum()
rolling_max = cum_return.rolling(20).max()
final_data['drawdown_20d'] = cum_return - rolling_max

downside_returns = final_data['market_return'].clip(upper=0)
final_data['downside_vol_20d'] = downside_returns.rolling(20).std()



In [14]:
final_data

,log_returns_AAPL,log_returns_JPM,log_returns_JNJ,log_returns_XOM,log_returns_WMT,log_returns_CAT,market_realized_vol_20d,market_cross_sectional_dispersion,leading_eigenvalue,second_eigenvalue,...,eigenvalue_gap_2_3,vol_of_vol_10d,market_return,market_return_skew_20d,vol_dispersion_corr_20d,return_autocorr_20d,vol_zscore_60d,dispersion_zscore_60d,drawdown_20d,downside_vol_20d
2016-12-01,-0.009363,0.020006,0.000718,-0.000688,0.003402,0.007091,0.134806,0.050412,2.498491,1.303026,...,0.581583,NaN,0.516799,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-12-02,0.003738,-0.002326,0.005194,-0.002295,0.002967,-0.011495,0.134647,0.051465,2.518076,1.312925,...,0.591437,NaN,0.517890,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-12-05,-0.007214,0.020139,-0.000178,0.005043,-0.013350,-0.007279,0.133923,0.051954,2.514906,1.332011,...,0.597162,NaN,0.518318,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-12-06,0.007669,0.005152,0.001071,0.000914,0.005987,0.008120,0.122621,0.044618,2.356767,1.379532,...,0.608394,NaN,0.501896,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-12-07,0.009775,0.004530,-0.008604,0.005807,0.010537,0.021917,0.123510,0.044839,2.336403,1.347861,...,0.545846,NaN,0.498853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-28,0.016145,0.009553,0.008505,0.001724,0.004472,-0.020715,0.124297,0.047196,1.809536,1.248599,...,0.109780,0.011497,0.295585,0.087688,0.770861,0.723317,0.631090,-0.158640,0.0,0.0
2026-08-31,-0.008955,-0.004484,-0.008204,0.026697,0.017119,-0.003480,0.124443,0.047862,1.843089,1.233418,...,0.074721,0.010662,0.300411,0.138500,0.766026,0.678471,0.653288,-0.054976,0.0,0.0
2026-09-01,0.025797,-0.003010,0.019887,0.022121,0.009963,-0.023228,0.116263,0.044146,1.848233,1.198723,...,0.061558,0.009785,0.302754,-0.135512,0.777488,0.623729,-0.284473,-0.669624,0.0,0.0
2026-09-02,-0.000523,0.003572,0.014715,-0.002434,0.001604,0.016699,0.117858,0.043068,1.908588,1.190760,...,0.064819,0.008783,0.311915,0.010539,0.784136,0.595144,-0.104968,-0.844577,0.0,0.0


# FEATURE SELECTION

In [15]:
feature_cols = ['market_realized_vol_20d', 'market_cross_sectional_dispersion',
                 'downside_vol_20d', 'leading_eigenvalue', 'eigenvalue_ratio',
                 'vol_of_vol_10d', 'eigenvalue_ratio_slope10', 'market_return_skew_20d',
                 'vol_zscore_60d', 'dispersion_zscore_60d',
                 'eigenvalue_ratio_diff5', 'vol_diff5', 'dispersion_diff5',
                 'vol_dispersion_corr_20d', 'vol_slope10']

In [16]:
feature_cols = ['market_realized_vol_20d', 'market_cross_sectional_dispersion',
                 'leading_eigenvalue', 'second_eigenvalue', 'eigenvalue_ratio',
                 'eigenvalue_ratio_diff5', 'vol_diff5', 'dispersion_diff5']

# **CLASSIFICATION MODEL**

In [17]:
from sklearn.model_selection import train_test_split as tts , GridSearchCV as grid ,RandomizedSearchCV as rand
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,VotingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,classification_report
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier

final features

In [18]:
train=final_data[:1800]
test=final_data[1800:]

In [19]:
x=train[feature_cols]
y=train['regime']

xtrain,xtest,ytrain,ytest=tts(x,y,shuffle=False,test_size=0.2)

In [20]:
model = VotingClassifier([
    ('rfc', RandomForestClassifier(random_state=42)),
    ('xgb', XGBClassifier(random_state=42)),
    ('lgb',LGBMClassifier(random_state=42))
], voting='hard')

In [21]:
model.fit(xtrain,ytrain)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000374 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 1440, number of used features: 8
[LightGBM] [Info] Start training from score -1.139000
[LightGBM] [Info] Start training from score -0.877137
[LightGBM] [Info] Start training from score -1.332227
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


VotingClassifier(estimators=[('rfc', RandomForestClassifier(random_state=42)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=True,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_polic...
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None, ...)),
                             ('lgb', LGBMClassifier(random_state=42))])

In [22]:
model.score(xtest,ytest)

0.8888888888888888

In [23]:
pred=model.predict(xtest)
round(accuracy_score(ytest,pred),2)

0.89

In [24]:
print(classification_report(ytest, model.predict(xtest)))

               precision    recall  f1-score   support

         calm       0.95      0.75      0.84       110
     stressed       0.98      0.96      0.97       146
transitioning       0.75      0.93      0.83       104

     accuracy                           0.89       360
    macro avg       0.89      0.88      0.88       360
 weighted avg       0.90      0.89      0.89       360



In [25]:
x=test[feature_cols]
y=test['regime']

pred=model.predict(x)
print((accuracy_score(y,pred)))

0.8788343558282209


In [26]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score
import numpy as np

X = final_data[feature_cols]
y = final_data['regime']

tscv = TimeSeriesSplit(n_splits=5)

scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    scores.append(acc)

    print(f"Fold {fold+1}: accuracy = {acc:.3f}, train size = {len(X_train)}, test size = {len(X_test)}")

print()
print(f"Mean accuracy: {np.mean(scores):.3f}")
print(f"Std deviation: {np.std(scores):.3f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000092 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1101
[LightGBM] [Info] Number of data points in the train set: 412, number of used features: 8
[LightGBM] [Info] Start training from score -0.347700
[LightGBM] [Info] Start training from score -1.703535
[LightGBM] [Info] Start training from score -2.192382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

# **PORTFOLIO OVERLAY**

**LAYER - 1**

**The covariance matrix**

This is the mathematical core of portfolio risk. Here's why you need it,
conceptually: a portfolio's total risk isn't just "add up each stock's individual risk" — it depends on how stocks move together. If two stocks always move in opposite directions, holding both actually reduces your combined risk (they cancel out); if they always move together, holding both doesn't reduce risk much at all. Covariance is the mathematical object that captures both each stock's own variance (on the diagonal) and every pair's co-movement (off-diagonal) — it's the input every portfolio optimization formula needs. .cov() computes this automatically from your historical returns; .values just converts it from a pandas DataFrame into a plain numpy array, which is the format CVXPY expects.

**Setting up the optimization variables**

cp.Variable(n) tells CVXPY: "there are 6 unknown numbers I want you to solve for." These 6 numbers (w) represent portfolio weights — what fraction of your total money sits in each stock. Right now they're undefined; the solver's job is to find the actual values.


**The objective — what "minimum variance" actually means mathematically**

This is the standard formula for portfolio variance: wᵀ Σ w (w-transpose times the covariance matrix times w). Intuitively: it takes your proposed weights, multiplies them through the covariance matrix, and produces a single number representing how much total risk that specific weight combination produces. cp.quad_form is just CVXPY's built-in shortcut for this exact formula, so you don't have to write out matrix multiplication by hand.

**The constraints — the "rules" the solution must obey**

`cp.sum(w) == 1` — your weights must add up to 100% of your capital (you're not leaving money uninvested, nor using leverage beyond your capital).
`w >= 0` — no negative weights, which means no short-selling — you can only buy stocks, not bet against them. This matches your project's design ("long-only min-variance," per the original plan).

**Solving it**

This tells CVXPY: "find the values of w that make portfolio_variance as small as possible, while still satisfying both constraints above." CVXPY runs a numerical optimization algorithm internally (you don't need to know its mechanics) and stores the answer.

**LAYER - 2**

Concept first: np.linalg.eigh (not eigvalsh) returns both eigenvalues AND their corresponding eigenvectors together. The leading eigenvector is a 6-dimensional direction — think of it as "if the whole market moves as one block, this vector describes each stock's relative contribution to that block movement." A portfolio that's very "aligned" with this direction is one that's heavily exposed to broad market co-movement; a portfolio that's more "orthogonal" to it is more diversified against that specific risk.
Code — extracting today's leading eigenvector (single snapshot, not the full rolling loop yet)



What this does: eigenvectors comes back as a matrix where each column is one eigenvector. Since eigh sorts eigenvalues smallest-to-largest (same convention as eigvalsh before), the very last column corresponds to the largest eigenvalue — the leading eigenvector. This 6-number vector is what we'll use to constrain the portfolio.
Run this and share the 6 values that come out — then I'll explain exactly how we turn this into a portfolio constraint (projecting the portfolio weights onto this direction, and capping that projection when the regime is transitioning/stressed).

In [35]:
import numpy as np
import cvxpy as cp
import pandas as pd

stock_names = universe

n = len(stock_cols)

# --- 1. Covariance matrix (for portfolio variance) ---
cov_matrix = final_data[stock_cols].cov().values

# --- 2. Correlation matrix + leading eigenvector (for the regime-aware constraint) ---
recent_window = final_data[stock_cols].iloc[-60:]
corr_matrix = recent_window.corr()

eigenvalues, eigenvectors = np.linalg.eigh(corr_matrix)
leading_eigenvector = eigenvectors[:, -1]   # eigh returns ascending order; last column = leading
v = leading_eigenvector

# --- 3. Baseline portfolio (no regime awareness) ---
w = cp.Variable(n)
portfolio_variance = cp.quad_form(w, cov_matrix)
constraints = [cp.sum(w) == 1, w >= 0]
problem = cp.Problem(cp.Minimize(portfolio_variance), constraints)
problem.solve()

baseline = w.value

# --- 4. Constrained portfolio (regime = stressed, cap exposure to leading eigenvector) ---
w3 = cp.Variable(n)
portfolio_variance3 = cp.quad_form(w3, cov_matrix)
exposure3 = w3 @ v

constraints3 = [
    cp.sum(w3) == 1,
    w3 >= 0,
    cp.abs(exposure3) <= 0.15   # tunable threshold
]
problem3 = cp.Problem(cp.Minimize(portfolio_variance3), constraints3)
problem3.solve()

constrained = w3.value

# --- 5. Comparison table ---
def classify_change(base, new):
    diff = new - base
    pct_of_base = diff / base if base != 0 else 0

    if abs(pct_of_base) < 0.05:
        return "~flat"
    elif pct_of_base >= 0.5:
        return "↑↑ (big jump)"
    elif pct_of_base > 0:
        return "↑"
    elif pct_of_base <= -0.5:
        return "↓↓ (big drop)"
    else:
        return "↓"

comparison = pd.DataFrame({
    'Stock': stock_names,
    'Baseline': np.round(baseline, 3),
    'Constrained': np.round(constrained, 3),
})
comparison['Change'] = [classify_change(b, c) for b, c in zip(baseline, constrained)]

print("Exposure (baseline):   ", baseline @ v)
print("Exposure (constrained):", constrained @ v)
print("Variance (baseline):   ", problem.value)
print("Variance (constrained):", problem3.value)
print()
print(comparison.to_string(index=False))

Exposure (baseline):    -0.3961673332602027
Exposure (constrained): -0.15000000000000002
Variance (baseline):    9.623675532219908e-05
Variance (constrained): 0.000113968198548766

Stock  Baseline  Constrained        Change
 AAPL     0.065        0.027 ↓↓ (big drop)
  JPM     0.055        0.160 ↑↑ (big jump)
  JNJ     0.446        0.329             ↓
  XOM     0.112        0.010 ↓↓ (big drop)
  WMT     0.285        0.242             ↓
  CAT     0.036        0.232 ↑↑ (big jump)


In [36]:
import yfinance as yf
import numpy as np
import pandas as pd

def latest_feature_extraction(tickers, window=60, vol_window=20):
    stock_cols = [f'log_returns_{t}' for t in tickers]

    # 1. Pull enough history to fill all rolling windows (60-day eigenvalue window + buffer)
    prices = yf.download(tickers, period='2y')['Close'][:-1]
    prices.columns = [f'log_returns_{t}' for t in prices.columns]

    # 2. Log returns
    data = np.log(prices / prices.shift(1)).dropna()

    # 3. Market return, vol, dispersion
    data['market_return'] = data[stock_cols].mean(axis=1)
    data['market_realized_vol_20d'] = data['market_return'].rolling(vol_window).std() * np.sqrt(252)
    data['market_cross_sectional_dispersion'] = data[stock_cols].std(axis=1)

    # 4. Rolling eigenvalues (leading, second, third) + ratio
    leading_eig, second_eig, third_eig, dates = [], [], [], []
    for i in range(window, len(data)):
        window_data = data[stock_cols].iloc[i-window:i]
        corr_matrix = window_data.corr()
        eigenvalues = np.linalg.eigvalsh(corr_matrix)
        eigenvalues_sorted = eigenvalues[::-1]
        leading_eig.append(eigenvalues_sorted[0])
        second_eig.append(eigenvalues_sorted[1])
        third_eig.append(eigenvalues_sorted[2])
        dates.append(data.index[i])

    eigen_df = pd.DataFrame({
        'leading_eigenvalue': leading_eig,
        'second_eigenvalue': second_eig,
        'third_eigenvalue': third_eig
    }, index=dates)
    eigen_df['eigenvalue_ratio'] = eigen_df['leading_eigenvalue'] / eigen_df['second_eigenvalue']

    data = data.merge(eigen_df, left_index=True, right_index=True, how='inner')

    # 5. Diff / slope / gap / vol-of-vol / skew / dispersion-corr / autocorr / z-scores / drawdown / downside vol
    def rolling_slope(series, w=10):
        return series.rolling(w).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0], raw=True)

    data['eigenvalue_ratio_diff5'] = data['eigenvalue_ratio'].diff(5)
    data['vol_diff5'] = data['market_realized_vol_20d'].diff(5)
    data['dispersion_diff5'] = data['market_cross_sectional_dispersion'].diff(5)
    data['eigenvalue_ratio_slope10'] = rolling_slope(data['eigenvalue_ratio'], 10)
    data['vol_slope10'] = rolling_slope(data['market_realized_vol_20d'], 10)
    data['eigenvalue_gap_2_3'] = data['second_eigenvalue'] - data['third_eigenvalue']
    data['vol_of_vol_10d'] = data['market_realized_vol_20d'].rolling(10).std()
    data['market_return_skew_20d'] = data['market_return'].rolling(20).skew()
    data['vol_dispersion_corr_20d'] = data['market_realized_vol_20d'].rolling(20).corr(data['market_cross_sectional_dispersion'])
    data['return_autocorr_20d'] = data['market_return'].rolling(20).apply(lambda x: pd.Series(x).autocorr(lag=1), raw=False)
    data['vol_zscore_60d'] = (data['market_realized_vol_20d'] - data['market_realized_vol_20d'].rolling(60).mean()) / data['market_realized_vol_20d'].rolling(60).std()
    data['dispersion_zscore_60d'] = (data['market_cross_sectional_dispersion'] - data['market_cross_sectional_dispersion'].rolling(60).mean()) / data['market_cross_sectional_dispersion'].rolling(60).std()

    cum_return = data['market_return'].cumsum()
    data['drawdown_20d'] = cum_return - cum_return.rolling(20).max()
    data['downside_vol_20d'] = data['market_return'].clip(upper=0).rolling(20).std()

    data = data.dropna()

    # 6. Return only the latest available row
    return data.iloc[[-1]]

In [37]:
tickers = universe
latest_row = latest_feature_extraction(tickers)
print(latest_row)

/tmp/ipykernel_2008/868230306.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(tickers, period='2y')['Close'][:-1]
[*********************100%***********************]  6 of 6 completed


            log_returns_AAPL  log_returns_CAT  log_returns_JNJ  \
2026-09-03          0.009952         0.009872         0.011632   

            log_returns_JPM  log_returns_WMT  log_returns_XOM  market_return  \
2026-09-03         0.016261         0.021725        -0.011889       0.009592   

            market_realized_vol_20d  market_cross_sectional_dispersion  \
2026-09-03                 0.122219                           0.011467   

            leading_eigenvalue  ...  vol_slope10  eigenvalue_gap_2_3  \
2026-09-03            1.909084  ...    -0.002349            0.072776   

            vol_of_vol_10d  market_return_skew_20d  vol_dispersion_corr_20d  \
2026-09-03        0.007821               -1.885462                 0.063058   

            return_autocorr_20d  vol_zscore_60d  dispersion_zscore_60d  \
2026-09-03             0.155684        0.385316              -0.728026   

            drawdown_20d  downside_vol_20d  
2026-09-03     -0.007182           0.00562  

[1 rows x 27 

In [38]:
latest_features=latest_row[feature_cols]

In [39]:
# Get the most recent row's features (or any specific date you want to check)
#latest_features = final_data[feature_cols].iloc[[-1]]   # double brackets keep it as a DataFrame

regime_today = model.predict(latest_features)[0]

print("Predicted regime:", regime_today)

# Same constraint logic as before, now driven by the real prediction
if regime_today in ['transitioning', 'stressed']:
    constraints3 = [
        cp.sum(w3) == 1,
        w3 >= 0,
        cp.abs(exposure3) <= 0.15
    ]
else:
    constraints3 = [
        cp.sum(w3) == 1,
        w3 >= 0
        # no exposure cap when regime is calm
    ]

problem3 = cp.Problem(cp.Minimize(portfolio_variance3), constraints3)
problem3.solve()

Predicted regime: transitioning


np.float64(0.000113968198548766)

In [40]:
problem3.solve()
print("Portfolio weights:", w3.value)
print("Portfolio variance:", problem3.value)

Portfolio weights: [0.02691662 0.15962744 0.32908587 0.00976785 0.24215061 0.2324516 ]
Portfolio variance: 0.00011396819854876596


In [41]:
latest_features

,market_realized_vol_20d,market_cross_sectional_dispersion,leading_eigenvalue,second_eigenvalue,eigenvalue_ratio,eigenvalue_ratio_diff5,vol_diff5,dispersion_diff5
2026-09-03,0.122219,0.011467,1.909084,1.197842,1.593769,0.098688,-0.00902,0.00397
